# 00 — Archive inventory and acquisition QC

Start here. Edit **analysis_workflow.json** once for all notebooks. The default is a reproducible four-file pilot, one recording day/file per May–August month. Set selection mode to `all` for full discovery; remove any file limit before interpreting archive-wide coverage. Every table/figure states its scope.

This reads original waveform samples sequentially. It does not alter the archive, calibrate counts or correct clocks. Triggered-file coverage is not station uptime, and file counts are not an unbiased earthquake rate. See **ANALYSIS_NOTEBOOKS.md** for the sequence and **METRIC_SCHEMA.md** for conventions.

Generated products use the shared `project_config.json` results root. Source/input files remain separate.


In [ ]:
from pathlib import Path
import sys
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in _candidates if (p / 'project_config.json').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start in the Pinatubo repository; project_config.json must be available.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from pinatubo_project import input_path, output_path, runtime_environment
runtime_environment()

from pathlib import Path
import sys
_candidates = [Path.cwd(), Path.cwd() / '20_analysis', *Path.cwd().parents]
ANALYSIS_DIR = next((p for p in _candidates if (p / 'analysis_workflow.py').is_file()), None)
if ANALYSIS_DIR is None:
    raise FileNotFoundError('Start in 20_analysis or its project root, beside analysis_workflow.py.')
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))
from analysis_workflow import load_context, describe_context
from IPython.display import display, Image, Markdown
import matplotlib.pyplot as plt
import pandas as pd
ctx = load_context(ANALYSIS_DIR)
display(describe_context(ctx))


## Inventory, read failures and sampling

The inventory retains auxiliary channels separately, checks missing samples, flat prefixes/suffixes, repeated extrema, record lengths and a narrowband 40.5-Hz screen. Repeated extrema suggest possible clipping; they do not establish ADC saturation without hardware limits. A narrowband peak does not identify its origin. A `not_testable` result is not an absence result. Zero-filled gaps inherited from conversion cannot reliably be recovered.

In [ ]:
from analysis_workflow import build_inventory
FORCE_REBUILD = False
inventory, files = build_inventory(ctx, force=FORCE_REBUILD)
display(files)
columns = ['trace_key', 'trace_id', 'is_waveform', 'sampling_rate_hz', 'duration_s',
           'missing_samples', 'flat_prefix_s', 'flat_suffix_s', 'repeated_extreme_run_samples',
           'artifact_status', 'artifact_detected', 'artifact_peak_hz', 'artifact_psd_ratio',
           'gain_epoch_id', 'epoch_status', 'qc_flags']
display(inventory[columns])
display(pd.read_csv(ctx['out'] / 'segment_gaps.csv'))


## Selected-record coverage and quality screens

Colours and typography are shared with subsequent notebooks. Figures are saved as 300-dpi PNG plus PDF/SVG, with a caption beside each. Sampling-rate changes, padding and gain epochs must be considered before comparing metrics.

In [ ]:
from analysis_workflow import plot_inventory
fig = plot_inventory(ctx, inventory)
display(Image(filename=str(ctx['out'] / 'figures/archive_inventory.png')))
plt.close(fig)
print('Inventory and provenance:', ctx['out'])


Next: **10_stationxml_and_network.ipynb**, then **20_pretrigger_and_window_selection.ipynb**. The older combined `00_archive_qc_and_pretrigger.ipynb` remains available as an earlier exploratory notebook; this sequence does not depend on it.